# Sensibilidade à prevalência

Visualização interna por reponderação de classes. Nenhum modelo ou calibrador é reajustado.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'results').is_dir() else Path.cwd().parent
RUN = ROOT / 'results/runs/20260721-023732_prevalence-k5_a3a584f3'
manifest = json.loads((RUN / 'manifest.json').read_text())
assert manifest['final_test_used'] is False
metrics = json.loads((RUN / 'metrics.json').read_text())


In [ ]:
rows = []
for item in metrics['targets']:
    balanced = item['policies']['balanced']
    rows.append({
        'prevalência': item['target_prevalence'],
        'probabilidade média': item['weighted_mean_probability'],
        'Brier': item['weighted_brier'],
        'ECE-10': item['weighted_ece'],
        'risco equilibrado': balanced['selective_risk'],
        'cobertura equilibrada': balanced['coverage'],
    })
frame = pd.DataFrame(rows)
frame.round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(frame['prevalência'], frame['ECE-10'], marker='o')
axes[0].axvline(metrics['observed_prevalence'], linestyle='--', color='0.5', label='observada')
axes[0].set(xlabel='Prevalência positiva', ylabel='ECE-10', title='Calibração')
axes[0].legend(frameon=False)
axes[1].plot(frame['prevalência'], frame['risco equilibrado'], marker='o', label='risco')
axes[1].plot(frame['prevalência'], frame['cobertura equilibrada'], marker='s', label='cobertura')
axes[1].set(xlabel='Prevalência positiva', title='Política equilibrada')
axes[1].legend(frameon=False)
for axis in axes: axis.grid(alpha=0.2)
fig.tight_layout();


A reponderação mantém as distribuições condicionais observadas. Ela não simula novos produtos, drift ou mudanças no retriever.